In [23]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

In [24]:
dataset = pd.read_csv("health_dataset.csv")
dataset
dataset1 = dataset

In [25]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Full_Name          500 non-null    object 
 1   Date_of_Birth      500 non-null    object 
 2   Email              500 non-null    object 
 3   Glucose_mg_dL      500 non-null    int64  
 4   Haemoglobin_g_dL   500 non-null    float64
 5   Cholesterol_mg_dL  500 non-null    int64  
 6   Remarks            0 non-null      float64
dtypes: float64(2), int64(2), object(3)
memory usage: 27.5+ KB


In [26]:
def split_scaler(indep_x, dep_y):
    x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.30, random_state = 0)
    sc = StandardScaler()
    x_train = sc.fit_transform(x_train)
    x_test = sc.transform(x_test)
    return x_train, x_test, y_train, y_test

In [27]:
def cm_prediction(grid, x_test, y_test):
    grid_pred = grid.predict(x_test)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, grid_pred)
    
    from sklearn.metrics import accuracy_score
    accuracy = accuracy_score(y_test, grid_pred)
    
    from sklearn.metrics import classification_report
    report = classification_report(y_test, grid_pred)
    return grid, accuracy, report, x_test, y_test, cm

In [28]:
# Functions collaborated with different algorithms

def logistic(x_train, y_train, x_test, y_test):
    param_grid = {'solver':['liblinear','saga'], 'penalty':['l2','l1']}
    grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3, n_jobs = -1,scoring = 'f1_weighted')
    grid.fit(x_train, y_train)
    classifier = LogisticRegression()
    classifier.fit(x_train, y_train)
    #coefficients = classifier.coef_[0]  # coef_[0]--weights of each feature; abs(coefficients)--- taking positive values even it is negative eg: -1.5=1.5
    #feature_imp_lr = pd.DataFrame({'Feature': feature_names,'Importance': abs(coefficients)}).sort_values(by='Importance', ascending=False)
    grid,accuracy,report,x_test,y_test,cm = cm_prediction(grid,x_test, y_test)
    return grid,accuracy,report,x_test,y_test,cm

def svm_linear(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    param_grid = {'kernel':['linear'], 'gamma':['scale', 'auto'],
              'decision_function_shape':['ovo', 'ovr']}
    grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
    grid.fit(x_train,y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid, accuracy, report, x_test, y_test, cm

def svm_nl(x_train, y_train, x_test, y_test):
    param_grid = {'kernel':['poly', 'rbf', 'sigmoid'], 'gamma':['scale', 'auto'],
              'decision_function_shape':['ovo', 'ovr']}
    grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
    grid.fit(x_train, y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid, accuracy, report, x_test, y_test, cm    

def knn(x_train, y_train, x_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    param_grid = {'algorithm':['auto', 'ball_tree', 'kd_tree', 'brute'], 'metric':['minkowski'],
              'weights':['uniform', 'distance']}
    grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
    grid.fit(x_train,y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid,accuracy,report,x_test,y_test,cm    

def naive(x_train, y_train, x_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]}
    grid = GridSearchCV(GaussianNB(), param_grid, cv=5, scoring = 'accuracy')
    grid.fit(x_train, y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid, accuracy, report, x_test, y_test, cm    

def decision(x_train, y_train, x_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    param_grid = {'criterion':['gini', 'entropy', 'log_loss'],'splitter':['best', 'random'],
                  'max_features':['sqrt', 'log2']}
    grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
    grid.fit(x_train,y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid, accuracy, report, x_test, y_test, cm    

def random(x_train, y_train, x_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    param_grid = {'criterion':['gini','entropy', 'log_loss'],'max_features':['sqrt', 'log2', None],
              'class_weight':['balanced', 'balanced_subsample'], 'min_samples_split':[1,2]}

    grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
    grid.fit(x_train,y_train)
    grid, accuracy, report, x_test, y_test, cm = cm_prediction(grid, x_test, y_test) 
    return grid, accuracy, report, x_test, y_test, cm    



In [29]:
dataset1.drop(['Full_Name', 'Date_of_Birth', 'Email', 'Remarks'], axis =1, inplace = True)
dataset1

,Glucose_mg_dL,Haemoglobin_g_dL,Cholesterol_mg_dL
0,133,9.5,167
1,176,10.7,192
2,193,15.3,303
3,164,17.2,149
4,117,13.4,255
...,...,...,...
495,72,9.4,207
496,190,10.0,292
497,75,13.8,166
498,156,16.3,319


In [30]:
def assign_disease_risk(row):
    glucose = row["Glucose_mg_dL"]
    haemoglobin = row["Haemoglobin_g_dL"]
    cholesterol = row["Cholesterol_mg_dL"]

    if glucose >= 126 and cholesterol >= 240:
        return "High Health Risk"

    elif glucose >= 126:
        return "Diabetes Risk"
        if random.random() < 0.1:
            disease = "Healthy"

    elif haemoglobin < 12:
        return "Anemia Risk"

    elif cholesterol >= 240:
        return "Heart Disease Risk"
     
    else:
        return "Healthy"

dataset1["Disease_Risk"] = dataset1.apply(assign_disease_risk, axis =1)
dataset1

,Glucose_mg_dL,Haemoglobin_g_dL,Cholesterol_mg_dL,Disease_Risk
0,133,9.5,167,Diabetes Risk
1,176,10.7,192,Diabetes Risk
2,193,15.3,303,High Health Risk
3,164,17.2,149,Diabetes Risk
4,117,13.4,255,Heart Disease Risk
...,...,...,...,...
495,72,9.4,207,Anemia Risk
496,190,10.0,292,High Health Risk
497,75,13.8,166,Healthy
498,156,16.3,319,High Health Risk


In [31]:
data1 = pd.get_dummies(dataset1, dtype = int, drop_first = True)
data1

,Glucose_mg_dL,Haemoglobin_g_dL,Cholesterol_mg_dL,Disease_Risk_Diabetes Risk,Disease_Risk_Healthy,Disease_Risk_Heart Disease Risk,Disease_Risk_High Health Risk
0,133,9.5,167,1,0,0,0
1,176,10.7,192,1,0,0,0
2,193,15.3,303,0,0,0,1
3,164,17.2,149,1,0,0,0
4,117,13.4,255,0,0,1,0
...,...,...,...,...,...,...,...
495,72,9.4,207,0,0,0,0
496,190,10.0,292,0,0,0,1
497,75,13.8,166,0,1,0,0
498,156,16.3,319,0,0,0,1


In [39]:
def selectkclassification(acclog, accsvml, accsvmnl, accknn, accnaive, accdeci, accrand):
    dataframe = pd.DataFrame(index = ['ChiSquare'], columns =['Logistic', 'SVML', 'SVMNL', 'KNN', 'Naive', 'Decision', 'Random'])
    for number,item in enumerate(dataframe.index):
        dataframe['Logistic'][item] = acclog[number]
        dataframe['SVML'][item] = accsvml[number]
        dataframe['SVMNL'][item] = accsvmnl[number]
        dataframe['KNN'][item] = accknn[number]
        dataframe['Naive'][item] = accnaive[number]
        dataframe['Decision'][item] = accdeci[number]
        dataframe['Random'][item] = accrand[number]
    return dataframe

In [40]:
# Input and Output Split

X = dataset1.drop("Disease_Risk", axis =1)
Y = dataset1["Disease_Risk"]

x_train, x_test, y_train, y_test = train_test_split(X, Y,test_size = 0.2, random_state = 32)

print(Y.shape)

(500,)


In [41]:

acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnaive = []
accdeci = []
accrand = []


In [42]:
grid, accuracy, report, x_test, y_test, cm = logistic(x_train, y_train, x_test, y_test)
acclog.append(accuracy)

grid, accuracy, report, x_test, y_test, cm = svm_linear(x_train, y_train, x_test, y_test)
accsvml.append(accuracy)

grid, accuracy, report, x_test, y_test, cm= svm_nl(x_train, y_train, x_test, y_test)
accsvmnl.append(accuracy)

grid, accuracy, report, x_test, y_test, cm = knn(x_train, y_train, x_test, y_test)
accknn.append(accuracy)

grid, accuracy, report, x_test, y_test, cm = naive(x_train, y_train, x_test, y_test)
accnaive.append(accuracy)

grid, accuracy, report, x_test, y_test, cm = decision(x_train, y_train, x_test, y_test)
accdeci.append(accuracy)

grid, accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
accrand.append(accuracy)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits


In [43]:
result = selectkclassification(acclog, accsvml, accsvmnl, accknn, accnaive, accdeci, accrand)

In [44]:
result

,Logistic,SVML,SVMNL,KNN,Naive,Decision,Random
ChiSquare,0.91,0.98,0.97,0.84,0.99,0.99,1.0


In [45]:
print(X.columns.tolist())

['Glucose_mg_dL', 'Haemoglobin_g_dL', 'Cholesterol_mg_dL']


In [46]:
print(dataset1["Disease_Risk"].value_counts())

Disease_Risk
Diabetes Risk         164
High Health Risk      102
Anemia Risk            89
Healthy                78
Heart Disease Risk     67
Name: count, dtype: int64


In [51]:
# Save the best model

import pickle

filename = "health_prediction_model.sav"

pickle.dump(grid, open(filename, "wb"))
loaded_model = pickle.load(open("health_prediction_model.sav", "rb"))
prediction = loaded_model.predict([[190, 13.5, 160]])

print(prediction)

['Diabetes Risk']


In [49]:
grid, accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
accrand.append(accuracy)
print(cm)
print(accuracy)

from sklearn.model_selection import cross_val_score

scores = cross_val_score(grid, X, Y, cv=5)

print(scores)
print("Mean Accuracy:", scores.mean())

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[[21  0  0  0  0]
 [ 0 28  0  0  0]
 [ 0  0 16  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 25]]
1.0
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
[0.98996129 1.         0.99006593 1.         0.98993514]
Mean Accuracy: 0.99399247190473
